In [ ]:
import sys
from pathlib import Path

# Project root is one level up from notebooks/
_proj_root = Path.cwd().parent
if str(_proj_root / 'src') not in sys.path:
    sys.path.insert(0, str(_proj_root / 'src'))

DATA_DIR = str(_proj_root / 'data')
print(f'Project root: {_proj_root}')
print(f'src in sys.path: {str(_proj_root / "src") in sys.path}')


In [1]:
"""
EVCP Pipeline — EVCP(5, 3, 3)
================================
5 POIs | 3 existing chargers | place 3 new chargers
q=8 → 256 cells (16×16)
"""

import math
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from tabulate import tabulate

from helpers import (
    divide_graph_into_parts,
    calculate_cell_weights,
    plot_cell_weights,
    print_weight_summary,
    suggest_parameters,
    print_parameter_suggestions,
)
from qubo_builder import (
    build_qubo,
    evaluate_solution,
    brute_force_ranking,
    print_qubo_diagnostics,
    print_service_gaps,
)

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Dataset
# ─────────────────────────────────────────────────────────────────────────────

points_of_interest = [
    (1,  15, 0.9),
    (2,  8,  0.6),
    (2,  6,  0.4),
    (7,  12, 0.8),
    (13, 6,  0.3),
]
existing_charging_points = [
    (3,  7),
    (8,  3),
    (11, 8),
]
gas_stations = [
    (5,  10),
    (10, 5),
    (3,  14),
    (9,  15),
]

M            = 3       # new chargers to place
NUM_QUBITS   = 8
X_MIN, X_MAX = 0, 14
Y_MIN, Y_MAX = 0, 19
SCALE_FACTOR = 5.0
MIN_WEIGHT   = 0.5

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Input data plot
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    [p[0] for p in points_of_interest],
    [p[1] for p in points_of_interest],
    c=[p[2] for p in points_of_interest],
    cmap=cm.YlOrRd, norm=mcolors.Normalize(vmin=0, vmax=1),
    s=150, zorder=5, edgecolors='black', linewidths=0.5
)
plt.colorbar(sc, ax=ax, pad=0.02, shrink=0.5, label='Population Density')
ex_x, ex_y = zip(*existing_charging_points)
ax.scatter(ex_x, ex_y, color='red', s=180, marker='*', zorder=5,
           edgecolors='black', linewidths=0.5, label='Existing Charger')
gs_x, gs_y = zip(*gas_stations)
ax.scatter(gs_x, gs_y, color='limegreen', s=100, marker='s', zorder=5,
           edgecolors='black', linewidths=0.5, label='Gas Station')
ax.scatter([], [], c='orange', s=100, edgecolors='black', linewidths=0.5, label='POI')
ax.set_xticks(range(0, 15))
ax.set_yticks(range(0, 20))
ax.grid(True, linewidth=0.5, alpha=0.3)
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.set_title('EVCP (5,3,3) — Input Data')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

/tmp/ipykernel_27854/1366885656.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Grid discretization
# ─────────────────────────────────────────────────────────────────────────────

grid_details, plot_deets = divide_graph_into_parts(
    x_min=X_MIN, x_max=X_MAX, y_min=Y_MIN, y_max=Y_MAX,
    num_qubits=NUM_QUBITS,
    points_of_interest=points_of_interest,
    existing_chargers=existing_charging_points,
    gas_stations=gas_stations,
    grid_division="default",
    grid_details_flag=True,
)

  Grid Info:
    Grid   41 (row=2, col=9): 1 charger(s)
    Grid   75 (row=4, col=11): 1 gas station(s)
    Grid   82 (row=5, col=2): 1 POI(s) [densities: 0.40]
    Grid   83 (row=5, col=3): 1 charger(s)
    Grid   94 (row=5, col=14): 1 POI(s) [densities: 0.30]
    Grid   98 (row=6, col=2): 1 POI(s) [densities: 0.60]
    Grid  108 (row=6, col=12): 1 charger(s)
    Grid  133 (row=8, col=5): 1 gas station(s)
    Grid  168 (row=10, col=8): 1 POI(s) [densities: 0.80]
    Grid  179 (row=11, col=3): 1 gas station(s)
    Grid  193 (row=12, col=1): 1 POI(s) [densities: 0.90]
    Grid  202 (row=12, col=10): 1 gas station(s)


/home/mepanda/Projects/Jayasri Ma'am/HGQA_Codespace/helpers.py:201: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Cell weight calculation
# ─────────────────────────────────────────────────────────────────────────────

cell_weights = calculate_cell_weights(
    grid_details, scale_factor=SCALE_FACTOR, min_weight=MIN_WEIGHT
)
print_weight_summary(cell_weights, grid_details, top_n=8)

plot_cell_weights(
    plot_deets, grid_details, cell_weights,
    points_of_interest=points_of_interest,
    existing_chargers=existing_charging_points,
    gas_stations=gas_stations,
    show_data_points=True, show_weight_values=False,
)
plt.savefig('plot_02_cell_weights.png', dpi=120)
plt.close()

CELL WEIGHT SUMMARY
------------------------  --------
Total cells               256
Cells with POIs           5 (2.0%)
Total aggregated density  3.00
Max weight                5.00
Min non-zero weight       1.67
Total gas stations        4
Total existing chargers   3
------------------------  --------

TOP 8 CELLS BY WEIGHT (Grid Data Table):
  Grid ID    Row    Col    # POIs    Raw    Norm    Weight    Gas Sta.    Chargers
---------  -----  -----  --------  -----  ------  --------  ----------  ----------
      193     12      1         1    0.9   1         5               0           0
      168     10      8         1    0.8   0.889     4.444           0           0
       98      6      2         1    0.6   0.667     3.333           0           0
       82      5      2         1    0.4   0.444     2.222           0           0
       94      5     14         1    0.3   0.333     1.667           0           0


/home/mepanda/Projects/Jayasri Ma'am/HGQA_Codespace/helpers.py:426: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. Parameter suggestion (FA-002)
# ─────────────────────────────────────────────────────────────────────────────

params = suggest_parameters(grid_details, cell_weights, plot_deets, m=M)
print_parameter_suggestions(params)

r, a, t    = params['radii'], params['alpha'], params['intra']
lambda_val = params['lambda']


SUGGESTED PARAMETERS  (FA-002)

Local Radii:
Radius                                      Value
----------------------------------------  -------
R₁  — H1 POI attraction radius                  4
Rₛ  — service gap radius (matches R₁)           4
R₃  — H3 existing charger penalty radius        2
R₄  — H4 new charger spacing radius             3
R₆  — H6 coverage redundancy radius             2

Objective Weights (α):
Parameter                                     Value
------------------------------------------  -------
α₁  — H1 POI attraction         [dominant]     3
α₂  — H2 gas station bonus                     0.5
α₃  — H3 existing charger penalty              1.57
α₄  — H4 new charger spacing                   1.5
α₅  — H5 constraint  (applied to λ)            1
α₆  — H6 coverage redundancy                   1.5

Intra-term Magnitudes:
Parameter                                  Value
---------------------------------------  -------
β  — gas station bonus magnitude               1
γ 

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. QUBO construction
# ─────────────────────────────────────────────────────────────────────────────

Q_obj, h5_params, diags = build_qubo(
    grid_details, cell_weights, plot_deets, m=M,
    alpha1=a['a1'], alpha2=a['a2'], alpha3=a['a3'],
    alpha4=a['a4'], alpha5=a['a5'], alpha6=a['a6'],
    beta=t['beta'], gamma=t['gamma'], delta=t['delta'], epsilon=t['epsilon'],
    lam=lambda_val,
    R1=r['R1'], Rs=r['Rs'], R3=r['R3'], R4=r['R4'], R6=r['R6'],
)

diag_count    = sum(1 for (i, j) in Q_obj if i == j)
offdiag_count = sum(1 for (i, j) in Q_obj if i != j)
diags['diag_entries']    = diag_count
diags['offdiag_entries'] = offdiag_count

print_qubo_diagnostics(diags)
print_service_gaps(diags)

N = diags['N']
print(f"\nQ_obj: {diags['total_Q_obj_entries']} entries  "
      f"({diag_count} diagonal + {offdiag_count} off-diagonal)  |  "
      f"Sparsity: {100*(1 - diags['total_Q_obj_entries']/N**2):.1f}%")

QUBO CONSTRUCTION DIAGNOSTICS (Q_obj — H1 through H6, H5 separate)

Grid:  N=256 cells,  m=3 new chargers
  POI cells:            5
  Cells w/ chargers:    3
  Individual chargers:  3  (E_all — counts: {41: 1, 83: 1, 108: 1})
  Radii: R1=4, Rs=4, R3=2, R4=3, R6=2

NORMALIZATION SCALES (raw max-abs, divisor applied before α):
  H1 : scale=5.00000  raw range [-5.0000, +0.0000]
  H2 : scale=1.00000  raw range [-1.0000, -0.0000]
  H3 : scale=1.00000  raw range [+0.0000, +1.0000]

DIAGONAL RANGES IN Q_obj (post-normalization × α):
  H1 : [-3.0000, +0.0000]
  H2 : [-0.5000, -0.0000]
  H3 : [+0.0000, +1.5700]
  H5: NOT stored in Q_obj

OFF-DIAGONAL PAIRS IN Q_obj:
  H4 (within R4=3):  4838 non-zero pairs
  H6 (within R6=2):  1090 non-zero pairs
  H5: NOT stored in Q_obj (use h5_params)

Q_obj SIZE:  5234 total entries  (diagonal: 256, off-diagonal: 4978)

SERVICE GAP FACTORS s_c  (Rs=4, 3 individual charger(s) across 3 cell(s)):
  Charger cells → counts: {41: 1, 83: 1, 108: 1}
  Cell  82 (row

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Solution evaluation — hand-picked comparisons (q=8)
# ─────────────────────────────────────────────────────────────────────────────

top_w        = sorted(cell_weights, key=lambda g: cell_weights[g]['weight'], reverse=True)
gas_cell_ids = [g for g, info in grid_details.items() if info['num_gas_stations'] > 0]
chr_cell_ids = [g for g, info in grid_details.items() if info['num_existing_chargers'] > 0]

solutions = [
    ("Top-3 weighted cells",      top_w[:3]),
    ("Spread: 0, N//2, N-1",      [0, N//2, N-1]),
    ("Best gas station cells",    sorted(gas_cell_ids,
                                         key=lambda g: cell_weights[g]['weight'],
                                         reverse=True)[:3]),
    ("Existing charger cells",    sorted(chr_cell_ids)[:3]),
    ("Worst-3 weighted cells",    top_w[-3:]),
]

rows = []
for label, sol in solutions:
    f_obj = evaluate_solution(Q_obj, sol)
    rows.append([label, sol, f"{f_obj:.4f}"])

print(f"\nSolution Comparison  (C({N},{M}) = {math.comb(N,M):,} — brute force skipped)")
print(tabulate(rows, headers=["Solution", "Cell IDs", "f_obj"], tablefmt="simple"))

best_label, best_sol = min(solutions, key=lambda x: evaluate_solution(Q_obj, x[1]))
print(f"\nBest: {best_label}  →  cells {best_sol}  "
      f"score {evaluate_solution(Q_obj, best_sol):.4f}")


Solution Comparison  (C(256,3) = 2,763,520 — brute force skipped)
Solution                Cell IDs           f_obj
----------------------  ---------------  -------
Top-3 weighted cells    [193, 168, 98]   -6.7383
Spread: 0, N//2, N-1    [0, 128, 255]    -1.2667
Best gas station cells  [75, 133, 179]   -2.9769
Existing charger cells  [41, 83, 108]     2.9044
Worst-3 weighted cells  [253, 254, 255]   2

Best: Top-3 weighted cells  →  cells [193, 168, 98]  score -6.7383


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Brute-force oracle — q=4 (N=16, m=2), all 120 solutions
# ─────────────────────────────────────────────────────────────────────────────

tiny_grid, tiny_plot = divide_graph_into_parts(
    x_min=X_MIN, x_max=X_MAX, y_min=Y_MIN, y_max=Y_MAX,
    num_qubits=4,
    points_of_interest=points_of_interest,
    existing_chargers=existing_charging_points,
    gas_stations=gas_stations,
    grid_division="default",
    grid_details_flag=True,
)
tiny_weights = calculate_cell_weights(tiny_grid, scale_factor=SCALE_FACTOR, min_weight=MIN_WEIGHT)
tiny_params  = suggest_parameters(tiny_grid, tiny_weights, tiny_plot, m=2)
tr, ta, tt   = tiny_params['radii'], tiny_params['alpha'], tiny_params['intra']

tiny_Q, tiny_h5, tiny_diags = build_qubo(
    tiny_grid, tiny_weights, tiny_plot, m=2,
    alpha1=ta['a1'], alpha2=ta['a2'], alpha3=ta['a3'],
    alpha4=ta['a4'], alpha5=ta['a5'], alpha6=ta['a6'],
    beta=tt['beta'], gamma=tt['gamma'], delta=tt['delta'], epsilon=tt['epsilon'],
    lam=tiny_params['lambda'],
    R1=tr['R1'], Rs=tr['Rs'], R3=tr['R3'], R4=tr['R4'], R6=tr['R6'],
)

N_tiny   = tiny_diags['N']
n_combos = math.comb(N_tiny, 2)
top10    = brute_force_ranking(tiny_Q, N_tiny, 2, top_k=10)

bf_rows = []
for rank, (score, sol) in enumerate(top10, 1):
    cells   = list(sol)
    r0, c0  = divmod(cells[0], tiny_plot['num_cols'])
    r1, c1  = divmod(cells[1], tiny_plot['num_cols'])
    has_poi = any(tiny_grid[c]['num_pois'] > 0 for c in cells)
    bf_rows.append([rank, cells, f"{score:.4f}",
                    f"(r={r0},c={c0})", f"(r={r1},c={c1})",
                    "✓" if has_poi else ""])

print(f"\nBrute-force Top 10  (q=4, N={N_tiny}, m=2, all {n_combos} solutions ranked)")
print(tabulate(bf_rows,
               headers=["Rank", "Cells", "Score", "Cell 0", "Cell 1", "Has POI"],
               tablefmt="simple"))